# Runner — Anatomy-Aware Pose Estimation (clean full run)

Pipeline completa: **baseline training** (MobileNetV3-Small + deconv head) →
**STL fine-tuning** (Skeletal Topology Loss con severity weighting, modello finale
`sev_E7`) → **valutazione** su COCO + OCHuman.

Diagnostiche, ablation e tuning esplorativi sono documentati in `README.md` e
`AVR_failure_analysis.md` — qui resta solo ciò che serve a riprodurre il risultato.

**Augmentation (Pytel vs Han):** imposta `AUGMENTATION=True` e `AUG_MODE` nel setup
per attivare l'occlusione sintetica in training (richiede `data.py` esteso).

In [9]:
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"

if os.path.isdir("/kaggle/working"):
    # === KAGGLE: clona il repo ===
    REPO_DIR = "/kaggle/working/repo"
    !rm -rf {REPO_DIR}
    !git clone -q {REPO_URL} {REPO_DIR}
    # trova la cartella con config.py
    mod_dir = None
    for root, dirs, files in os.walk(REPO_DIR):
        if ".git" in root: continue
        if "config.py" in files:
            mod_dir = root
            break
else:
    # === LOCALE: il notebook è già dentro anatomy-aware-pose/ ===
    mod_dir = os.getcwd()

if mod_dir and os.path.exists(os.path.join(mod_dir, "config.py")):
    sys.path.insert(0, mod_dir)
    os.chdir(mod_dir)
    print("Working dir:", os.getcwd())
    print("Moduli:", sorted([f for f in os.listdir(mod_dir) if f.endswith(".py")]))
else:
    print("ERRORE: config.py non trovato in", mod_dir)

Working dir: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose
Moduli: ['anthropometric_constraints.py', 'config-local.py', 'config.py', 'data.py', 'evaluation.py', 'network.py', 'stl.py', 'test_stl.py', 'train.py', 'utils.py']


In [10]:
# === 2. SETUP ESPERIMENTO — interruttori della run ===
# Cambia QUI cosa eseguire, senza toccare il resto del notebook.

# --- AUGMENTATION (Pytel vs Han) ---
AUGMENTATION = False           # True per attivare l'occlusione sintetica in training
AUG_MODE     = "limb_segment"  # 'random' | 'limb_segment' | 'limb_joint'  (se AUGMENTATION)
AUG_PROB     = 0.5             # prob. di occludere un'immagine
AUG_NUM      = 1               # quante occlusioni per immagine

# --- TRAINING ---
TRAIN_BASELINE = False         # True per ri-allenare la baseline da zero (30 ep, lungo)
TRAIN_STL      = True          # True per il fine-tuning STL dal best.pth

# --- STL fine-tuning (modello finale sev_E7) ---
STL_LR     = 1e-5              # LR vincente (config A)
STL_BOOST  = 5.0              # boost su lambda_bone calibrato
STL_EPOCHS = 8                # sev_E7 = epoch 7; 8 per avere margine di selezione

# la severity e' in stl.py (SEVERITY_ALPHA=2.0). Per ablation hinge: settarla a 0 nel file.
_aug = AUG_MODE if AUGMENTATION else "none"
print(f"AUGMENTATION={AUGMENTATION} (mode={_aug}, prob={AUG_PROB}, num={AUG_NUM})")
print(f"TRAIN_BASELINE={TRAIN_BASELINE} | TRAIN_STL={TRAIN_STL}")
print(f"STL: LR={STL_LR}, boost={STL_BOOST}x, epochs={STL_EPOCHS}")

AUGMENTATION=False (mode=none, prob=0.5, num=1)
TRAIN_BASELINE=False | TRAIN_STL=True
STL: LR=1e-05, boost=5.0x, epochs=8


In [11]:
# === 3. Import + seed + device ===
import importlib
import numpy as np
import torch
from torch.utils.data import DataLoader

import config, data, network, train, evaluation, utils, stl
for m in (config, data, network, train, evaluation, utils, stl):
    importlib.reload(m)
from config import *           # path, iperparametri, set_seed, device, ENV_NAME, BEST_PTH
import evaluation as ev

set_seed(SEED)
print(f"Ambiente: {ENV_NAME} | device: {device} | seed: {SEED}")
print(f"SEVERITY_ALPHA = {stl.SEVERITY_ALPHA} (0=hinge, 2=modello finale)")
print(f"best.pth: {BEST_PTH}  ({'OK' if os.path.exists(BEST_PTH) else 'ASSENTE'})")

Ambiente: LOCAL | device: cuda | seed: 42
SEVERITY_ALPHA = 2.0 (0=hinge, 2=modello finale)
best.pth: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth  (OK)


In [12]:
import inspect
print(inspect.signature(data.COCOKeypointsDataset.__init__))

(self, samples, img_dir, input_size, heatmap_size, sigma, num_kpts, aug_mode='none', aug_prob=0.5, aug_num=1, seed=42)


In [13]:
# === 4. Dati COCO (+ OCHuman per la valutazione) ===
g = torch.Generator(); g.manual_seed(SEED)

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"COCO train: {len(train_samples)} | val: {len(val_samples)}")

# dataset di training: con augmentation se attiva (GT sempre intatta)
train_dataset = data.COCOKeypointsDataset(
    train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS,
    aug_mode=_aug, aug_prob=AUG_PROB, aug_num=AUG_NUM, seed=SEED)
val_dataset = data.COCOKeypointsDataset(
    val_samples, COCO_VAL_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)  # no aug in val

# deterministico: num_workers=0 + generator seedato
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=0, pin_memory=True, generator=g)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False,
                          num_workers=0, pin_memory=True)
print(f"Augmentation in training: {_aug}")

loading annotations into memory...
Done (t=3.89s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
COCO train: 116021 | val: 6352
Augmentation in training: none


In [14]:
# === 5. Training BASELINE (opzionale: TRAIN_BASELINE=True) ===
# MobileNetV3-Small + deconv head, 30 epoche, WeightedMSELoss. Produce best.pth.
# Di norma gia' fatto (best.pth nel repo); riallenare solo se serve (es. baseline+aug).
if TRAIN_BASELINE:
    NUM_EPOCHS = 30
    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = train.WeightedMSELoss()
    history = train.fit(model, train_loader, val_loader, optimizer, scheduler,
                        criterion, device, NUM_EPOCHS, CHECKPOINT_DIR, resume=True)
    del model, optimizer; torch.cuda.empty_cache()
    print("Baseline training completato -> best.pth")
else:
    print("TRAIN_BASELINE=False: uso best.pth esistente.")

TRAIN_BASELINE=False: uso best.pth esistente.


In [15]:
# === 6. Fine-tuning STL (modello finale, severity in stl.py) ===
# Parte da best.pth, calibra i lambda sulla norma del gradiente (vede la loss
# severity gia' attiva in stl.py), applica boost su bone, seleziona su AVR.
import time
from stl import SkeletalTopologyLoss, calibrate_lambdas

if TRAIN_STL:
    import stl as _stl_mod  # per toccare SEVERITY_ALPHA temporaneamente

    set_seed(SEED)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader_det = DataLoader(
        train_dataset, batch_size=32, shuffle=True,
        num_workers=0, pin_memory=True, generator=g)

    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    model.load_state_dict(torch.load(BEST_PTH, map_location=device))
    print(f"Pesi baseline caricati da {BEST_PTH}")

    criterion = SkeletalTopologyLoss(
        heatmap_criterion=train.WeightedMSELoss(),
        lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
        beta=STL_BETA)

    # Calibra sulla hinge pura (come sev_E7): spegni severity solo qui
    _orig_alpha = _stl_mod.SEVERITY_ALPHA
    _stl_mod.SEVERITY_ALPHA = 0.0
    print("Calibrazione lambda (gradient-norm, severity OFF)...")
    calibrate_lambdas(criterion, model, train_loader_det, device,
                      target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)
    _stl_mod.SEVERITY_ALPHA = _orig_alpha   # ripristina per il training
    print(f"SEVERITY_ALPHA ripristinato a {_orig_alpha}")

    lam_calib = criterion.lambda_bone
    criterion.lambda_bone = lam_calib * STL_BOOST   # STL_BOOST = 5.0
    print(f"lambda_bone: {lam_calib:.5f} -> {criterion.lambda_bone:.5f} ({STL_BOOST}x)\n")

    optimizer = torch.optim.Adam(model.parameters(), lr=STL_LR)
    optimizer.zero_grad(set_to_none=True)
    tag = f"stl_{_aug}" if AUGMENTATION else "stl_final"
    OUT_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", tag))
    os.makedirs(OUT_DIR, exist_ok=True)

    model.eval()
    _, avr0 = ev.evaluate_on_coco_val(model, val_samples, device, results_path=f"{OUT_DIR}/pred_e00.json")
    pc0 = avr0['per_category']
    print(f"{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  {'L_bone':>8}  time  note")
    print("-"*80)
    print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} {pc0['bone_ratio']:>7.4f} "
          f"{pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}   (baseline E0)")

    best = (1e9, None)
    for epoch in range(1, STL_EPOCHS + 1):
        t0 = time.time(); model.train()
        sums = {k: 0.0 for k in ['heatmap','bone','total']}
        for imgs, hms, w in train_loader_det:
            imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
            optimizer.zero_grad()
            loss, terms = criterion(model(imgs), hms, w)
            loss.backward(); optimizer.step()
            for k in sums: sums[k] += terms[k] * imgs.size(0)
        n = len(train_loader_det.dataset)
        for k in sums: sums[k] /= n
        torch.save(model.state_dict(), f"{OUT_DIR}/epoch_{epoch:02d}.pth")
        model.eval()
        ce, avr = ev.evaluate_on_coco_val(model, val_samples, device, results_path=f"{OUT_DIR}/pred_e{epoch:02d}.json")
        ap = float(ce.stats[0]); pc = avr['per_category']; ar = avr['AVR_pose_rate']
        note = ""
        if ar < best[0] and ap >= 0.486: best = (ar, epoch); note = " *best"
        print(f"{epoch:>3} {ap:>7.4f} {ar:>7.4f} {pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} "
              f"{pc['collapse']:>7.4f}  {sums['bone']:>8.4f}  {time.time()-t0:.0f}s{note}")
    print(f"\nMiglior AVR: {best[0]:.4f} @ epoch {best[1]} -> {OUT_DIR}/epoch_{best[1]:02d}.pth")
    STL_BEST_PTH = f"{OUT_DIR}/epoch_{best[1]:02d}.pth"
else:
    print("TRAIN_STL=False: salto il fine-tuning.")

Pesi baseline caricati da /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth
Calibrazione lambda (gradient-norm, severity OFF)...
Calibrazione lambda (gradient-norm, rho=0.1, 4 batch, ref=final_layer):
  norme grad grezze: heatmap=2.103e-04  bone=1.856e-01  angle=2.409e-02  order=2.501e+00  collapse=4.987e-04
  lambda calibrati : bone=0.00011  angle=0.00087  order=0.00010  collapse=0.00194
SEVERITY_ALPHA ripristinato a 2.0
lambda_bone: 0.00011 -> 0.00057 (5.0x)



inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.10s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.499
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.485
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.522
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.799
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.577
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

...

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=20.49s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.785
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.484
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | m

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.497
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.785
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.531
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.522
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.574
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.776
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.531
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.484
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.522
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.798
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

In [16]:
# === 7. Valutazione finale: baseline vs STL su COCO + OCHuman (deterministica) ===
from evaluation import run_coco_eval, evaluate_avr, build_samples
from data import COCOEvalDataset
from utils import decode_heatmaps, heatmap_to_original
set_seed(SEED)

def _infer_det(model, samples, img_dir, bs=32):
    ds = COCOEvalDataset(samples, img_dir, INPUT_SIZE)
    loader = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0, pin_memory=True)
    model.eval(); res, coords, scores = [], [], []
    with torch.no_grad():
        for imgs, ids, bboxes in loader:
            imgs = imgs.to(device); hm = model(imgs)
            ch, sc = decode_heatmaps(hm); bn = bboxes.numpy(); idn = ids.numpy()
            for i in range(imgs.shape[0]):
                ci = heatmap_to_original(ch[i], bn[i], INPUT_SIZE, HEATMAP_SIZE); ks = sc[i]
                flat = []
                for k in range(NUM_KEYPOINTS): flat += [float(ci[k,0]), float(ci[k,1]), float(ks[k])]
                res.append({'image_id':int(idn[i]),'category_id':1,'keypoints':flat,'score':float(ks.mean())})
                coords.append(ci); scores.append(ks)
    return res, coords, scores

def evaluate(model, tag):
    r,c,s = _infer_det(model, val_samples, COCO_VAL_IMG)
    ce_c = run_coco_eval(r, COCO_VAL_ANN, f"/tmp/{tag}_coco.json"); avr_c = evaluate_avr(c,s)
    oc = build_samples(OCHUMAN_VAL_ANN)
    r,c,s = _infer_det(model, oc, OCHUMAN_IMG)
    ce_o = run_coco_eval(r, OCHUMAN_VAL_ANN, f"/tmp/{tag}_oc.json"); avr_o = evaluate_avr(c,s)
    return {'coco_ap':float(ce_c.stats[0]),'coco_avr':avr_c['AVR_pose_rate'],
            'oc_ap':float(ce_o.stats[0]),'oc_avr':avr_o['AVR_pose_rate'],'oc_pc':avr_o['per_category']}

def _load(p):
    m = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    m.load_state_dict(torch.load(p, map_location=device)); m.eval(); return m

rows = {}
rows['baseline'] = evaluate(_load(BEST_PTH), 'baseline')
if TRAIN_STL:
    rows['STL'] = evaluate(_load(STL_BEST_PTH), 'stl')

print("\n" + "="*70)
print(f"{'model':>10} | {'COCO AP':>8} {'COCO AVR':>9} | {'OC AP':>7} {'OC AVR':>8}")
print("-"*70)
for n,r in rows.items():
    print(f"{n:>10} | {r['coco_ap']:>8.4f} {r['coco_avr']:>9.4f} | {r['oc_ap']:>7.4f} {r['oc_avr']:>8.4f}")
if 'STL' in rows:
    b,sv = rows['baseline'], rows['STL']
    print("-"*70)
    print(f"{'delta':>10} | {'':>8} {sv['coco_avr']-b['coco_avr']:>+9.4f} | {'':>7} {sv['oc_avr']-b['oc_avr']:>+8.4f}")
    print("\nper-categoria OCHuman:")
    for n in ['baseline','STL']:
        pc = rows[n]['oc_pc']
        print(f"  {n:>8}: bone {pc['bone_ratio']:.4f}  angle {pc['joint_angle']:.4f}  collapse {pc['collapse']:.4f}")
print("="*70)

loading annotations into memory...
Done (t=20.57s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.11s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.799
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.575
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | m